# Vision Transformer (ViT) — Flower Classification Pro

A production-style image classification project built on two complementary Vision Transformer implementations:

1. **`vit_scratch.py`** — a Vision Transformer built entirely from `nn.Module` primitives (patch embedding via strided conv, multi-head self-attention, MLP blocks, learnable class token and position embeddings) — implementing the architecture described in *An Image is Worth 16x16 Words* (Dosovitskiy et al., 2020).
2. **`vit_transfer.py`** — a fine-tuned `torchvision.models.vit_b_16` pretrained on ImageNet-1k, with the classifier head replaced and the last transformer blocks unfrozen for fine-tuning.

**Dataset:** 3,670 images across 5 flower classes (daisy, dandelion, roses, sunflowers, tulips) — the widely-used TensorFlow `flower_photos` dataset, released under CC-BY 2.0.

| Class | Images |
|---|---|
| daisy | 633 |
| dandelion | 898 |
| roses | 641 |
| sunflowers | 699 |
| tulips | 799 |

Run this notebook top to bottom on a GPU runtime (Colab: Runtime → Change runtime type → GPU) for full training in minutes.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "src")

import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import prepare_data
import engine
import utils
from vit_scratch import vit_tiny, vit_base
from vit_transfer import build_pretrained_vit, unfreeze_last_n_blocks

utils.set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 2. Prepare the dataset

Stratified 70/15/15 train/val/test split, generated fresh from the raw class folders.

In [ ]:
summary = prepare_data.split_dataset(
    source_dir="data/flowers_raw",
    output_dir="data/flowers_split",
    train_ratio=0.7,
    val_ratio=0.15,
    seed=42,
)
summary

## 3. Data pipeline with augmentation

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder("data/flowers_split/train", transform=train_transform)
val_data = datasets.ImageFolder("data/flowers_split/val", transform=eval_transform)
test_data = datasets.ImageFolder("data/flowers_split/test", transform=eval_transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_data.classes
class_names, len(train_data), len(val_data), len(test_data)

## 4. Model A — ViT built from scratch

Trains the hand-built `ViT` architecture directly on the flower dataset, no pretrained weights. This is the honest, educational baseline: Vision Transformers have no convolutional inductive bias, so they need either a lot of data or pretraining to generalize well — this run demonstrates that limitation directly.

In [ ]:
scratch_model = vit_tiny(num_classes=len(class_names)).to(device)

optimizer = torch.optim.AdamW(scratch_model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
loss_fn = nn.CrossEntropyLoss()

scratch_results = engine.train(
    model=scratch_model,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=20,
    device=device,
    scheduler=scheduler,
    checkpoint_path="checkpoints/vit_scratch_best.pth",
)

In [ ]:
utils.plot_loss_curves(scratch_results, save_path="outputs/scratch_loss_curves.png")

## 5. Model B — Fine-tuned pretrained ViT-B/16

Loads ImageNet-pretrained `vit_b_16`, replaces the classification head, and fine-tunes the last two transformer blocks plus the new head. This is the model that will actually perform well on 3,670 images — pretraining on ImageNet gives it the visual priors that the from-scratch model has to learn from nothing.

In [ ]:
transfer_model, weights_transform = build_pretrained_vit(num_classes=len(class_names), freeze_backbone=True)
unfreeze_last_n_blocks(transfer_model, n=2)
transfer_model = transfer_model.to(device)

train_data_tf = datasets.ImageFolder("data/flowers_split/train", transform=weights_transform)
val_data_tf = datasets.ImageFolder("data/flowers_split/val", transform=weights_transform)
test_data_tf = datasets.ImageFolder("data/flowers_split/test", transform=weights_transform)

train_loader_tf = DataLoader(train_data_tf, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_tf = DataLoader(val_data_tf, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_tf = DataLoader(test_data_tf, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer_tf = torch.optim.AdamW(filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler_tf = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_tf, T_max=15)

transfer_results = engine.train(
    model=transfer_model,
    train_dataloader=train_loader_tf,
    val_dataloader=val_loader_tf,
    optimizer=optimizer_tf,
    loss_fn=loss_fn,
    epochs=15,
    device=device,
    scheduler=scheduler_tf,
    checkpoint_path="checkpoints/vit_transfer_best.pth",
)

In [ ]:
utils.plot_loss_curves(transfer_results, save_path="outputs/transfer_loss_curves.png")

## 6. Evaluation on the held-out test set

In [ ]:
scratch_model.load_state_dict(torch.load("checkpoints/vit_scratch_best.pth", map_location=device))
print("=== From-scratch ViT ===")
utils.evaluate_model(scratch_model, test_loader, class_names, device, save_path="outputs/scratch_confusion_matrix.png")

In [ ]:
transfer_model.load_state_dict(torch.load("checkpoints/vit_transfer_best.pth", map_location=device))
print("=== Fine-tuned ViT-B/16 ===")
utils.evaluate_model(transfer_model, test_loader_tf, class_names, device, save_path="outputs/transfer_confusion_matrix.png")

## 7. Attention visualization

Where is the from-scratch ViT actually looking? Visualizing the CLS token's attention over image patches in the last transformer block.

In [ ]:
import glob
sample_image = glob.glob("data/flowers_split/test/sunflowers/*")[0]
utils.visualize_attention(scratch_model, sample_image, eval_transform, device)

## 8. Single-image inference

In [ ]:
test_image = glob.glob("data/flowers_split/test/tulips/*")[0]
utils.predict_image(transfer_model, test_image, class_names, weights_transform, device)

## 9. Results summary

Fill in after a full GPU run:

| Model | Params (trainable) | Test Accuracy | Notes |
|---|---|---|---|
| ViT from scratch | ~5-6M | — | No pretraining, trained only on 3,670 images |
| Fine-tuned ViT-B/16 | ~1.5M (head + last 2 blocks) | — | ImageNet pretrained, transfer learning |
